# S-Channel Maze: Wavefront Optimization

Demonstrates **open eigenchannel routing** through a PEC cylinder maze.

An S-shaped channel is carved out of a dense array of PEC cylinders.
Normal incidence scatters off the walls, but the **SVD-optimal wavefront**
(last right singular vector of S11) finds the open channel and tunnels through.

**What this example demonstrates:**
1. Building a maze geometry with PEC cylinder walls
2. Normal incidence: most of the wave is reflected by the maze walls
3. Optimal wavefront: SVD of S11 reveals the open eigenchannel

In [ ]:
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import svd

sys.path.insert(0, '../..')

from Scattering_Code.smatrix_parameters import smatrix_parameters
from Scattering_Code.smatrix import smatrix
from Scattering_Code.ky import ky

## 1. Build S-Channel Maze Geometry

Place PEC cylinders in a grid pattern with an S-shaped gap carved out.

In [ ]:
WAVELENGTH = 0.93
PERIOD     = 12.0
RADIUS     = 0.2
MU         = 1.0
CMMAX      = 5
PHIINC     = np.pi / 2

# Grid of cylinders
spacing = 0.6
nx_grid = int(PERIOD / spacing)
ny_grid = 20
thickness = ny_grid * spacing

# Generate grid positions
all_positions = []
for iy in range(ny_grid):
    for ix in range(nx_grid):
        x = (ix + 0.5) * spacing
        y = (iy + 0.5) * spacing
        all_positions.append([x, y])

all_positions = np.array(all_positions)

# Carve S-channel: remove cylinders within channel_width of the S-path
channel_width = 1.2

def s_channel_center(y, period, thickness):
    """S-shaped channel center x-position as function of y."""
    t = y / thickness  # normalize to [0, 1]
    return period/2 + (period/4) * np.sin(2 * np.pi * t)

keep = []
for pos in all_positions:
    x, y = pos
    cx = s_channel_center(y, PERIOD, thickness)
    if abs(x - cx) > channel_width / 2:
        keep.append(pos)

clocs = np.array(keep)
num_cyl = len(clocs)
print(f"Maze: {num_cyl} PEC cylinders, thickness = {thickness}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 10))
theta = np.linspace(0, 2*np.pi, 30)
for i in range(num_cyl):
    ax.fill(clocs[i,0] + RADIUS*np.cos(theta),
            clocs[i,1] + RADIUS*np.sin(theta),
            color='gray', alpha=0.6, edgecolor='black', lw=0.3)

# Draw S-channel path
y_path = np.linspace(0, thickness, 200)
x_path = s_channel_center(y_path, PERIOD, thickness)
ax.plot(x_path, y_path, 'r--', lw=2, label='Channel center')

ax.axhline(0, color='magenta', lw=2)
ax.axhline(thickness, color='magenta', lw=2)
ax.set_xlim(-0.5, PERIOD + 0.5)
ax.set_ylim(-1, thickness + 1)
ax.set_aspect('equal')
ax.set_title(f'S-Channel Maze ({num_cyl} PEC cylinders)')
ax.legend()
plt.tight_layout()
plt.savefig('s_channel_maze_geometry.png', dpi=150)
plt.show()

## 2. Compute S-Matrix

In [ ]:
nmax = int(np.floor(PERIOD / WAVELENGTH))  # PEC
nm   = 2 * nmax + 1

sp = smatrix_parameters(WAVELENGTH, PERIOD, PHIINC,
                        1e-11, 1e-4, 5, 3, 1000, 3, 5, 1, PERIOD/120)
cmmaxs = np.full(num_cyl, CMMAX, dtype=int)
cepmus = np.column_stack([np.full(num_cyl, -1.0), np.full(num_cyl, MU)])
crads  = np.full(num_cyl, RADIUS)

print(f"Computing S-matrix ({num_cyl} cylinders, nmax={nmax})...")
t0 = time.time()
S, _ = smatrix(clocs, cmmaxs, cepmus, crads, PERIOD, WAVELENGTH,
               nmax, thickness, sp, 'On')
print(f"Done in {time.time()-t0:.1f}s")

## 3. Compare Normal Incidence vs. Optimal Wavefront

In [ ]:
S11 = S[:nm, :nm]
S21 = S[nm:, :nm]

# Normal incidence
Input_norm = np.zeros(nm, dtype=complex)
Input_norm[nmax] = 1.0
T_norm = np.sum(np.abs(S21 @ Input_norm)**2)

# Optimal wavefront
U, s, Vh = svd(S11)
Input_opt = Vh.conj().T[:, -1]
T_opt = np.sum(np.abs(S21 @ Input_opt)**2)

# Transmission eigenvalues
tau = svd(S21, compute_uv=False)

print(f"Normal incidence transmission:  {T_norm*100:.2f}%")
print(f"Optimal wavefront transmission: {T_opt*100:.2f}%")
print(f"Enhancement factor: {T_opt/T_norm:.1f}x")
print(f"\nmax(tau^2) = {np.max(tau**2):.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, len(tau)+1), tau**2, color='navy')
ax.axhline(1.0, color='red', ls='--', lw=1)
ax.set_xlabel('Channel index')
ax.set_ylabel(r'$\tau^2$')
ax.set_title('Transmission Eigenvalues — S-Channel Maze')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('maze_singular_values.png', dpi=150)
plt.show()